In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_timestamp

# 1. NEW DATA (Staging updates)
# Imagine this data comes from an external regulatory API
new_sanctions_data = [
    ("ENT_001", "NORTH_STAR_SHIPPING", "CRITICAL"), # Existing, no change
    ("ENT_002", "GLOBAL_LOGISTICS", "CRITICAL"),    # Existing, risk escalated (High -> Critical)
    ("ENT_004", "SHADOW_BANK_X", "HIGH")            # Brand new entity
]

df_updates = spark.createDataFrame(new_sanctions_data, ["entity_id", "entity_name", "risk_level"])

# 2. TARGET TABLE
target_path = "`prism-sentinel-stream`.prism_silver.sanctions_master"
target_table = DeltaTable.forName(spark, target_path)

# 3. IDENTIFY UPDATES
# Find existing records that need to be "closed" (because the risk level changed)
df_to_close = df_updates.alias("updates").join(
    spark.table(target_path).alias("target"),
    (col("updates.entity_id") == col("target.entity_id")) & (col("target.is_current") == True)
).filter("updates.risk_level <> target.risk_level") \
 .select("target.*") \
 .withColumn("valid_to", current_timestamp()) \
 .withColumn("is_current", lit(False))

# 4. PERFORM THE MERGE
# Step A: Close the old records
if df_to_close.count() > 0:
    target_table.alias("t").merge(
        df_to_close.alias("u"),
        "t.entity_id = u.entity_id AND t.valid_from = u.valid_from"
    ).whenMatchedUpdate(set={
        "valid_to": "u.valid_to",
        "is_current": "u.is_current"
    }).execute()

# Step B: Insert the new records (both brand new and updated versions)
df_to_insert = df_updates.withColumn("is_current", lit(True)) \
                         .withColumn("valid_from", current_timestamp()) \
                         .withColumn("valid_to", lit(None).cast("timestamp"))

df_to_insert.write.format("delta").mode("append").saveAsTable(target_path)

print("✅ Sanctions Master synchronized with SCD Type 2 history.")